# Featured Quotes for Artists

Get one random sample for `source_actor == "artist"` point per artist from `artist_points_flat.json` and write the result to a csv - will overwrite later based on manual curation

In [1]:
import json
import random
from pathlib import Path

import pandas as pd

In [2]:
artist_points_path = Path("../../data/clusters/artist_points_flat.json")
output_path = Path("../../data/addl/featured_quotes.csv")

with artist_points_path.open(encoding="utf-8") as f:
    artist_points = json.load(f)

artist_points_by_artist = {
    str(entry["artist_id"]): entry["points"]
    for entry in artist_points
}

In [3]:
rng = random.Random(42)

rows = []
for artist_id in sorted(artist_points_by_artist, key=lambda value: int(value)):
    points = [
        point
        for point in artist_points_by_artist.get(artist_id, [])
        if point.get("source_actor") == "artist" and not point.get("is_artwork")
    ]
    if not points:
        rows.append(
            {
                "artist_id": artist_id,
                "source_idx": pd.NA,
                "point": pd.NA,
                "text": pd.NA,
            }
        )
        continue

    selected = rng.choice(points)
    rows.append(
        {
            "artist_id": artist_id,
            "source_idx": selected.get("source_idx", pd.NA),
            "point": selected.get("id", pd.NA),
            "text": selected.get("text", pd.NA),
        }
    )

featured_quotes = pd.DataFrame(rows, columns=["artist_id", "source_idx", "point", "text"])

missing_points = int(featured_quotes["point"].isna().sum())
if missing_points:
    print(f"did not find featured quotes for {missing_points} artists")

featured_quotes[["artist_id", "source_idx", "point"]].to_csv(output_path, index=False)
display(featured_quotes.head(20))

,artist_id,source_idx,point,text
0,0,AT53,0,To be a human being as well as an artist is di...
1,1,AT01,5,Locating undiscovered territory or dimensional...
2,2,AT09,13,"You already mentioned Lolita, which deals with..."
3,3,AT18,18,"The genre of self-portraiture, especially when..."
4,4,AT25,29,Just as 100% Hapa isn’t really about race—it’s...
5,5,AT37,46,And I think that’s hard to explain to people. ...
6,6,AT41,52,"That was also right after the L.A. riots, and ..."
7,7,AT58,67,"For me, I’ve always had an estranged relations..."
8,8,AT69,76,"You’re talking to other human beings, having a..."
9,9,AT71,82,"Through making the documentary Wildness, I had..."
